In [15]:

print(os.getcwd())

/Users/aryamanghaisas/SyntheticDataPipeline


In [10]:
# DP Wrapper sanity check — 2 epochs, 1000 rows, ε=10
import os
os.chdir('SyntheticDataPipeline')  # ensure working dir is correct

In [17]:

import importlib, sys


# Clear cache to pick up dp_wrapper
to_remove = [key for key in sys.modules if key.startswith('src')]
for key in to_remove:
    del sys.modules[key]

import yaml
from src.generators.ctgan_generator import CTGANGenerator
from src.generators.tvae_generator import TVAEGenerator
from src.generators.dp_wrapper import DPWrapper

with open('config.yaml') as f:
    config = yaml.safe_load(f)

# Override epochs to 2 for speed
config['generators']['ctgan']['epochs'] = 2
config['generators']['tvae']['epochs']  = 2

DATASET = 'acs_income'   # cleanest dataset, fastest to run

for gen_cls, gen_name in [(CTGANGenerator, 'ctgan'), (TVAEGenerator, 'tvae')]:
    print(f'\n{"-"*50}')
    print(f'DP sanity check — {gen_name} × {DATASET} × ε=10')
    print(f'{"-"*50}')

    gen    = gen_cls(config, DATASET)
    dp_gen = DPWrapper(gen, epsilon=10.0, config=config)

    train_df = dp_gen.load_train_data()
    train_df = dp_gen.prepare_dataframe(train_df)
    train_df = train_df.sample(n=1000, random_state=42)   # small slice
    metadata = dp_gen.build_metadata(train_df)

    dp_gen.fit(train_df, metadata)
    synthetic = dp_gen.sample(100)

    print(f'Synthetic shape: {synthetic.shape}')
    print(f'NaN columns: {synthetic.isnull().sum()[synthetic.isnull().sum() > 0].to_dict() or "None"}')
    print(f'[PASSED] {gen_name} DP sanity check complete')


--------------------------------------------------
DP sanity check — ctgan × acs_income × ε=10
--------------------------------------------------


/Users/aryamanghaisas/SyntheticDataPipeline/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:138: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Gen. (+02.66) | Discrim. (-00.05): 100%|██████████| 2/2 [00:00<00:00,  8.04it/s]
/Users/aryamanghaisas/SyntheticDataPipeline/.venv/lib/python3.11/site-packages/opacus/accountants/analysis/rdp.py:332: UserWarning: Optimal order is the largest alpha. Please consider expanding the range of alphas to get a tighter privacy bound.
  warnings.warn(
CTGAN DP (ε=10.0): 100%|██████████| 2/2 [00:00<00:00, 17.13it/s]


Synthetic shape: (100, 11)
NaN columns: None
[PASSED] ctgan DP sanity check complete

--------------------------------------------------
DP sanity check — tvae × acs_income × ε=10
--------------------------------------------------


/Users/aryamanghaisas/SyntheticDataPipeline/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:138: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Loss: +45.25: 100%|██████████| 2/2 [00:00<00:00, 43.96it/s]
/Users/aryamanghaisas/SyntheticDataPipeline/.venv/lib/python3.11/site-packages/opacus/privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
/Users/aryamanghaisas/SyntheticDataPipeline/.venv/lib/python3.11/site-packages/opacus/accountants/analysis/rdp.py:332: UserWarning: Optimal order is the largest alpha. Please consider expanding the range of alphas to get a tighter privacy bound.
  warnings.warn(
08/03/2026 09:42:47:WARNING:Ignoring drop_last as it is not compatible with

Synthetic shape: (100, 11)
NaN columns: None
[PASSED] tvae DP sanity check complete


In [18]:
from src.generators.copulagan_generator import CopulaGANGenerator

gen    = CopulaGANGenerator(config, 'acs_income')
dp_gen = DPWrapper(gen, epsilon=10.0, config=config)
train_df = dp_gen.load_train_data()
train_df = dp_gen.prepare_dataframe(train_df)
train_df = train_df.sample(n=1000, random_state=42)
metadata = dp_gen.build_metadata(train_df)
dp_gen.fit(train_df, metadata)
synthetic = dp_gen.sample(100)
print(f'Synthetic shape: {synthetic.shape}')
print(f'NaN columns: {synthetic.isnull().sum()[synthetic.isnull().sum() > 0].to_dict() or "None"}')
print('[PASSED] copulagan DP sanity check complete')

/Users/aryamanghaisas/SyntheticDataPipeline/.venv/lib/python3.11/site-packages/sdv/single_table/base.py:138: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Gen. (-00.55) | Discrim. (+00.12): 100%|██████████| 300/300 [00:18<00:00, 16.29it/s]
CTGAN DP (ε=10.0): 100%|██████████| 300/300 [00:18<00:00, 16.50it/s]

Synthetic shape: (100, 11)
NaN columns: None
[PASSED] copulagan DP sanity check complete
